Подготовка датасета для CPT

In [ ]:
import os
import json
import random
from datasets import load_dataset

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
OUTPUT_DIR = "./cpt_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "cpt_corpus.jsonl")

Источник 1: StackExchange (ServerFault, NetworkEngineering) ArmelR/stack-exchange-instruction

In [ ]:
def load_stackexchange(max_samples=30000):
    texts = []
    try:
        ds = load_dataset(
            "ArmelR/stack-exchange-instruction",
            split="test",
            streaming=True,  # обязательно, иначе падает
        )
        for i, item in enumerate(ds):
            if len(texts) >= max_samples:
                break

            question = item.get("question", "").strip()
            response = item.get("response", "").strip()

            if not question or not response:
                continue

            #объединяем вопрос и ответ как единый текст
            text = f"{question}\n\n{response}"

            #базовая фильтрация: достаточно содержательный
            if len(text) < 200:
                continue

            #обрезаем слишком длинные
            if len(text) > 4000:
                text = text[:4000]

            texts.append(text)

            if len(texts) % 5000 == 0:
                print(f"  Загружено {len(texts)}")

        print(f"{len(texts)} текстов")
        return texts

    except Exception as e:
        print(f"Ошибка StackExchange: {e}")
        return []

In [ ]:
stackexchange_texts = load_stackexchange(max_samples=30000)

Источник 2: 3GPP Телекоммуникационные стандарты dinho1597/3GPP-Documents-100cs (1 столбец)

In [ ]:
def load_3gpp(max_samples=10000):
    texts = []

    try:
        ds = load_dataset("dinho1597/3GPP-Documents-100cs", split="train")

        #определяем поле с текстом
        text_field = None
        for candidate in ["text", "content", "document", "chunk"]:
            if candidate in ds.column_names:
                text_field = candidate
                break

        #если не нашли стандартное поле — берём первое
        if text_field is None:
            text_field = ds.column_names[0]

        print(f"  Используем поле: '{text_field}'")

        indices = list(range(len(ds)))
        random.shuffle(indices)

        for idx in indices:
            if len(texts) >= max_samples:
                break

            text = str(ds[idx][text_field]).strip()

            if len(text) < 100:
                continue
            if len(text) > 4000:
                text = text[:4000]

            texts.append(text)

        print(f"{len(texts)} текстов")
        return texts

    except Exception as e:
        print(f"Ошибка 3GPP: {e}")
        return []

In [ ]:
gpp_texts = load_3gpp(max_samples=10000)

Источник 3: smollm-corpus

In [ ]:
def load_tech_corpus(max_samples=20000):
    texts = []

    try:
        ds = load_dataset(
            "HuggingFaceTB/smollm-corpus",
            "cosmopedia-v2",
            split="train",
            streaming=True,   # датасет огромный (39M примеров), streaming обязателен
        )

        for item in ds:
            if len(texts) >= max_samples:
                break

            text = item.get("text", "").strip()

            if len(text) < 200:
                continue
            if len(text) > 4000:
                text = text[:4000]

            texts.append(text)

            if len(texts) % 5000 == 0:
                print(f"  Загружено {len(texts)} текстов...")

        print(f"cosmopedia-v2: {len(texts)} текстов")
        return texts

    except Exception as e:
        print(f"Ошибка cosmopedia-v2: {e}")
        return []


In [ ]:
tech_texts = load_tech_corpus(max_samples=20000)

Источник 4: Logpai/Loghub — системные логи (не HDFS)

In [ ]:
def load_loghub_bgl(max_samples=20000):
    texts = []

    sources = [
        ("vaibhav2507/bgl-logs", "train", "textstring"),
        ("Kingslayer5437/BGL", "train", "log"),
    ]

    for dataset_name, split, text_field in sources:
        if texts:
            break
        print(f"  Пробуем {dataset_name}...")
        try:
            ds = load_dataset(dataset_name, split=split, streaming=True)

            for item in ds:
                if len(texts) >= max_samples:
                    break

                text = item.get(text_field, "").strip()

                if not text or len(text) < 20:
                    continue
                if len(text) > 1000:
                    text = text[:1000]

                texts.append(text)

                if len(texts) % 5000 == 0:
                    print(f"  Загружено {len(texts)} текстов...")

            print(f"BGL логи ({dataset_name}): {len(texts)} текстов")

        except Exception as e:
            print(f"{dataset_name}: {e}")

    if not texts:
        print("Не удалось загрузить BGL логи")

    return texts

In [ ]:
log_texts = load_loghub_bgl(max_samples=20000)

Сборка единого файла

In [ ]:
def build_cpt_corpus():
    all_texts = []

    #загрузка каждого источника
    stackexchange_texts = load_stackexchange(max_samples=30000)
    gpp_texts = load_3gpp(max_samples=10000)
    tech_texts = load_tech_corpus(max_samples=20000)
    log_texts = load_loghub_bgl(max_samples=20000)

    all_texts.extend(stackexchange_texts)
    all_texts.extend(gpp_texts)
    all_texts.extend(tech_texts)
    all_texts.extend(log_texts)

    #перемешивание
    random.shuffle(all_texts)

    print(f"ИТОГО: {len(all_texts)}")
    print(f"  StackExchange: {len(stackexchange_texts)}")
    print(f"  3GPP:          {len(gpp_texts)}")
    print(f"  Tech-corpus:   {len(tech_texts)}")
    print(f"  Logs (BGL):    {len(log_texts)}")

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for text in all_texts:
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

    print(f"Сохранено: {OUTPUT_PATH}")
    print(f"Размер: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB")

    return OUTPUT_PATH

In [ ]:
corpus_path = build_cpt_corpus()